In [1]:
import torch

from ipynb.fs.defs.data import monte_carlo_data
from ipynb.fs.defs.features import build_features
from ipynb.fs.defs.q_network import QNetwork
from ipynb.fs.defs.save_load import load_model

In [2]:
model = load_model()

ACTION_MAP = {
    0: "HOLD",
    1: "BUY_50",
    2: "BUY_100",
    3: "SELL_50",
    4: "SELL_100"
}

Model loaded from models/dqn_trader.pt


In [3]:
def predict_signal(
    ticker,
    position_fraction=0.0,
    unrealized_pnl=0.0
):

    prices = monte_carlo_data(
        ticker,
        "2023-01-01",
        "2025-12-31"
    ).squeeze()

    features = build_features(prices)

    latest = features.iloc[-1]

    state = [
        latest["momentum"],
        latest["volatility"],
        latest["ma_signal"],
        latest["rsi"],
        latest["return_20"],
        latest["return_50"],
        latest["ma50_signal"],
        latest["ma200_signal"],
        latest["atr_proxy"],
        unrealized_pnl,
        position_fraction
    ]

    model = load_model()

    state_tensor = torch.tensor(
        state,
        dtype=torch.float32
    ).unsqueeze(0)

    with torch.no_grad():

        q_values = model(state_tensor).squeeze()
    
        valid_actions = [0, 1, 2, 3, 4]
        
        if position_fraction <= 0:
        
            valid_actions.remove(3)
            valid_actions.remove(4)
        
        elif position_fraction >= 0.99:
        
            valid_actions.remove(1)
            valid_actions.remove(2)
        
        best_action = valid_actions[0]
        best_q = q_values[best_action]
        
        for action in valid_actions:
        
            if q_values[action] > best_q:
        
                best_q = q_values[action]
                best_action = action
        
        valid_q = torch.tensor(
            [q_values[a] for a in valid_actions]
        )
        
        valid_probs = torch.softmax(
            valid_q,
            dim=0
        )
        
        confidence = float(
            valid_probs[
                valid_actions.index(best_action)
            ]
        )
        
        print("Q Values:", q_values)
        print("Valid Actions:", valid_actions)
        
        return {
            "ticker": ticker,
            "action": ACTION_MAP[best_action],
            "price": float(latest["price"])
        }